# 🧠 NomadGo — تدريب نموذج YOLOv8 مخصص للبقالة / المطاعم / الكافيهات

**المدة:** 60-90 دقيقة على GPU مجاني (T4)

**النتيجة:** ملف `nomadgo_inventory.onnx` جاهز للتثبيت في تطبيق NomadGo

---
### خطوات سريعة:
1. افتح هذا الدفتر في Google Colab
2. فعّل GPU: **Runtime → Change runtime type → T4 GPU**
3. شغّل كل الخلايا بالترتيب (`Ctrl+F9`)
4. حمّل ملف `.onnx` في النهاية

## الخلية 1 — تثبيت المكتبات

In [ ]:
!pip install ultralytics roboflow onnx onnxsim onnxruntime -q
import ultralytics
ultralytics.checks()
print('✅ المكتبات جاهزة')

## الخلية 2 — اختر مصدر البيانات

**الخيار A (مُوصى به):** بيانات Roboflow Universe — مجتمعة ومنوّعة للبقالة والمطاعم  
**الخيار B:** بيانات COCO المفلترة — سريع بدون تسجيل  

غيّر `DATA_SOURCE` حسب رغبتك:

In [ ]:
# ─── اختر المصدر ───────────────────────────────────────────────────────────────
DATA_SOURCE = 'ROBOFLOW'   # خيّر: 'ROBOFLOW' أو 'COCO_FILTER'

# إذا اخترت ROBOFLOW، ضع API key من https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'

# ─── إعدادات التدريب ───────────────────────────────────────────────────────────
EPOCHS      = 100    # عدد دورات التدريب (100 كافية، زد لـ 200 لدقة أعلى)
IMG_SIZE    = 640    # حجم الصورة (لا تغيّر)
BATCH_SIZE  = 16     # حجم الدفعة (خفّض لـ 8 إذا نفدت الذاكرة)
MODEL_BASE  = 'yolov8n.pt'  # n=nano (أسرع وأصغر للموبايل)

print(f'مصدر البيانات: {DATA_SOURCE}')
print(f'دورات التدريب: {EPOCHS}')

## الخلية 3أ — تحميل بيانات Roboflow (إذا اخترت ROBOFLOW)

In [ ]:
if DATA_SOURCE == 'ROBOFLOW':
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    # مجموعة بيانات الرفوف الغذائية من Roboflow Universe
    # تغطي: زجاجات، علب، صناديق، أكياس، منتجات البقالة
    project = rf.workspace('roboflow-100').project('grocery-store-efryh')
    dataset = project.version(2).download('yolov8')
    DATA_YAML = dataset.location + '/data.yaml'
    print(f'✅ بيانات Roboflow محمّلة في: {DATA_YAML}')

## الخلية 3ب — بناء بيانات COCO مفلترة (إذا اخترت COCO_FILTER)

In [ ]:
if DATA_SOURCE == 'COCO_FILTER':
    import os, json, shutil
    from pathlib import Path
    
    # تحميل COCO 2017 (الجزء الصغير للتجربة)
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O val2017.zip
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O annotations.zip
    !unzip -q val2017.zip && unzip -q annotations.zip
    
    # فئات مخصصة للبقالة والمطاعم من COCO
    TARGET_COCO_CATEGORIES = {
        44:  'bottle',       # زجاجة
        47:  'cup',          # كوب
        46:  'wine_glass',   # كأس
        51:  'bowl',         # طبق/وعاء
        50:  'banana',       # موز
        51:  'apple',        # تفاح
        55:  'sandwich',     # ساندويش
        56:  'orange',       # برتقال
        57:  'broccoli',     # بروكلي
        58:  'carrot',       # جزر
        59:  'hot_dog',      # هوت دوج
        60:  'pizza',        # بيتزا
        61:  'donut',        # دونات
        62:  'cake',         # كيك
        73:  'laptop',       # جهاز (للكافيهات)
        76:  'keyboard',     # لوحة مفاتيح
    }
    
    print('📦 فلترة COCO للفئات المطلوبة...')
    
    # بناء مجلد البيانات
    base = Path('nomadgo_dataset')
    for split in ['train', 'val']:
        (base / split / 'images').mkdir(parents=True, exist_ok=True)
        (base / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    with open('annotations/instances_val2017.json') as f:
        coco = json.load(f)
    
    cat_map = {c['id']: i for i, c in enumerate(coco['categories']) 
               if c['id'] in TARGET_COCO_CATEGORIES}
    custom_labels = [TARGET_COCO_CATEGORIES[c['id']] for c in coco['categories'] 
                     if c['id'] in TARGET_COCO_CATEGORIES]
    
    img_info = {img['id']: img for img in coco['images']}
    img_anns = {}
    for ann in coco['annotations']:
        if ann['category_id'] in cat_map:
            img_anns.setdefault(ann['image_id'], []).append(ann)
    
    count = 0
    for img_id, anns in img_anns.items():
        img = img_info[img_id]
        src = f"val2017/{img['file_name']}"
        if not os.path.exists(src): continue
        
        split = 'val' if count % 10 == 0 else 'train'
        dst_img = base / split / 'images' / img['file_name']
        dst_lbl = base / split / 'labels' / img['file_name'].replace('.jpg', '.txt')
        shutil.copy(src, dst_img)
        
        W, H = img['width'], img['height']
        lines = []
        for ann in anns:
            cls_id = cat_map[ann['category_id']]
            x, y, w, h = ann['bbox']
            cx, cy = (x + w/2) / W, (y + h/2) / H
            nw, nh = w/W, h/H
            lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        
        dst_lbl.write_text('\n'.join(lines))
        count += 1
    
    DATA_YAML = str(base / 'data.yaml')
    (base / 'data.yaml').write_text(
        f"path: {base.absolute()}\n"
        f"train: train/images\nval: val/images\n"
        f"nc: {len(custom_labels)}\n"
        f"names: {custom_labels}\n"
    )
    print(f'✅ تم إنشاء {count} صورة للتدريب')
    print(f'الفئات: {custom_labels}')

## الخلية 4 — إضافة بيانات مخصصة (اختياري)
إذا كانت لديك صور خاصة بمنتجاتك، ضعها هنا

In [ ]:
# ─── اختياري: رفع صورك الخاصة ─────────────────────────────────────────────────
# من Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/my_products /content/nomadgo_dataset/train/images/

# أو من جهازك مباشرة:
# from google.colab import files
# uploaded = files.upload()  # ارفع صور المنتجات

print('تخطّ هذه الخلية إذا لم يكن لديك صور خاصة')

## الخلية 5 — تدريب النموذج 🚀

In [ ]:
from ultralytics import YOLO

print(f'🚀 بدء التدريب على {EPOCHS} دورة...')
print('سيستغرق هذا 60-90 دقيقة على GPU T4')
print('─' * 50)

model = YOLO(MODEL_BASE)

results = model.train(
    data       = DATA_YAML,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    device     = 'cuda',     # GPU
    project    = 'nomadgo',
    name       = 'inventory_model',
    patience   = 30,         # إيقاف مبكر إذا لم يتحسن
    save       = True,
    plots      = True,
    # تحسينات للموبايل
    optimizer  = 'AdamW',
    lr0        = 0.001,
    lrf        = 0.01,
    warmup_epochs = 3,
    # data augmentation
    mosaic     = 1.0,
    mixup      = 0.1,
    flipud     = 0.0,
    fliplr     = 0.5,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
)

print('\n✅ اكتمل التدريب!')
print(f'أفضل دقة mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

## الخلية 6 — اختبار النموذج

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import glob

# أفضل نموذج محفوظ
best_model_path = sorted(glob.glob('nomadgo/inventory_model*/weights/best.pt'))[-1]
print(f'النموذج: {best_model_path}')

model = YOLO(best_model_path)

# تقييم على بيانات التحقق
metrics = model.val()
print(f'\n📊 نتائج التقييم:')
print(f'  mAP50:    {metrics.box.map50:.3f}')
print(f'  mAP50-95: {metrics.box.map:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}')
print(f'  Recall:    {metrics.box.mr:.3f}')

## الخلية 7 — تصدير إلى ONNX 📦

In [ ]:
from ultralytics import YOLO
import glob, shutil

best_model_path = sorted(glob.glob('nomadgo/inventory_model*/weights/best.pt'))[-1]
model = YOLO(best_model_path)

print('⚙️ تصدير إلى ONNX...')

# تصدير بدقة float16 (أسرع على الموبايل)
onnx_path = model.export(
    format    = 'onnx',
    imgsz     = 640,
    simplify  = True,     # تبسيط الرسم البياني
    opset     = 11,       # متوافق مع OnnxRuntime على Android
    dynamic   = False,    # batch size ثابت = 1
    half      = False,    # float32 (أكثر توافقاً مع Android)
    device    = 'cpu',
)

# إعادة تسمية الملف
import os
final_path = 'nomadgo_inventory.onnx'
shutil.copy(str(onnx_path), final_path)

size_mb = os.path.getsize(final_path) / (1024*1024)
print(f'\n✅ تم التصدير: {final_path}')
print(f'   الحجم: {size_mb:.1f} MB')

## الخلية 8 — تحسين وتحقق من ONNX

In [ ]:
import onnx
from onnxsim import simplify

# تحميل وتحقق
model_onnx = onnx.load('nomadgo_inventory.onnx')
onnx.checker.check_model(model_onnx)
print('✅ النموذج صحيح')

# تحسين إضافي
model_simplified, check = simplify(model_onnx)
if check:
    onnx.save(model_simplified, 'nomadgo_inventory.onnx')
    print('✅ تم التبسيط والتحسين')

# اختبار سريع
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('nomadgo_inventory.onnx')
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
output = session.run(None, {session.get_inputs()[0].name: dummy})
print(f'✅ اختبار الاستدلال ناجح — شكل الإخراج: {output[0].shape}')
print(f'\nإدخال النموذج: {session.get_inputs()[0].name}')
print(f'إخراج النموذج: {session.get_outputs()[0].name}')

## الخلية 9 — حفظ ملف labels.txt وتحميل كل شيء

In [ ]:
import yaml

# استخراج الفئات من ملف التدريب
with open(DATA_YAML) as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])

# حفظ labels.txt
with open('labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

print(f'✅ labels.txt يحتوي على {len(class_names)} فئة:')
for i, name in enumerate(class_names):
    print(f'  {i:2d}: {name}')

print('\n📥 تحميل الملفات...')

In [ ]:
from google.colab import files
import os

print('📥 جاهز للتحميل!')
print(f'nomadgo_inventory.onnx: {os.path.getsize("nomadgo_inventory.onnx")/1024/1024:.1f} MB')

# تحميل ملف ONNX
files.download('nomadgo_inventory.onnx')

# تحميل ملف labels
files.download('labels.txt')

print('\n✅ تم! الآن:')
print('1. ضع nomadgo_inventory.onnx في: Assets/StreamingAssets/Models/')
print('2. ضع labels.txt في: Assets/Resources/Models/')
print('3. أعد بناء APK من GitHub Actions')

## الخلية 10 — (اختياري) حفظ على Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('nomadgo_inventory.onnx', '/content/drive/MyDrive/NomadGo/nomadgo_inventory.onnx')
# shutil.copy('labels.txt', '/content/drive/MyDrive/NomadGo/labels.txt')
# print('✅ محفوظ على Google Drive')

print('أزل التعليق (#) من الأسطر أعلاه لحفظ على Google Drive')

---
## 📋 كيفية تثبيت النموذج في التطبيق

بعد تحميل الملفين:

### 1. ضع الملفات في المشروع
```
Assets/
├── StreamingAssets/
│   └── Models/
│       └── nomadgo_inventory.onnx   ← هنا
└── Resources/
    └── Models/
        └── labels.txt               ← هنا
```

### 2. عدّل CONFIG.json
```json
{
  "model": {
    "path": "Models/nomadgo_inventory.onnx",
    "labels_path": "Models/labels.txt",
    "confidence_threshold": 0.40
  }
}
```

### 3. أضف ONNX_RUNTIME في Unity
- **Edit → Project Settings → Player**
- في **Scripting Define Symbols** أضف: `ONNX_RUNTIME`
- ثم رفع للـ GitHub → يبني تلقائياً

---
*NomadGo SpatialVision — نموذج مخصص للعدّ الذكي*